# 03 — Evaluación del sistema GraphRAG (metodología RAGAS)

Este notebook evalúa el pipeline usando las tres métricas **RAGAS** descritas en
clase. El enfoque **no** requiere la librería RAGAS —
todas las métricas se implementan como llamadas estructuradas al LLM.

### Las tres métricas

| Métrica | Qué mide | Cómo |
|---|---|---|
| **context_recall** | ¿El recuperador obtuvo chunks que cubren la verdad esperada? | Cada frase del ground truth se verifica: ¿puede atribuirse al contexto recuperado? |
| **faithfulness** | ¿La respuesta contiene solo cosas respaldadas por el contexto? | La respuesta se descompone en afirmaciones atómicas; cada una se verifica contra el contexto. |
| **answer_correctness** | ¿Cuánto se superpone la respuesta con el ground truth? | Las afirmaciones se clasifican en TP / FP / FN; se calcula el F1. |

### El pipeline de evaluación

```
CSV de benchmark
  (question ; cypher)
        │
        ▼
  load_dataset()       ← lee el CSV, ejecuta Cypher para obtener ground truths dinámicos
        │
        ▼
  run_benchmark()      ← llama al agente RAG para cada pregunta, registra la latencia
        │
        ▼
  evaluate_results()   ← puntúa cada fila con las tres métricas RAGAS
        │
        ▼
  print_summary()      ← tabla agregada
```

**Requisito previo:** Ejecutar `01_ingestion_demo.ipynb` para cargar los datos de Einstein en Neo4j.

## 1. Configuración

In [1]:
import sys
sys.path.append('..')

import pandas as pd
# from graphrag.graph.neo4j_manager import Neo4jManager
from neo4j_manager import Neo4jManager
# from graphrag.agents import AgenticRAG
from agents.multi_agent_system import MultiAgentSystem
# from graphrag.evaluation import RAGEvaluator
from evaluator import RAGEvaluator

pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:.3f}'.format)

In [2]:
neo4j = Neo4jManager()
rag = MultiAgentSystem(neo4j)
evaluator = RAGEvaluator(rag, neo4j)
print("Sistema listo.")


Sistema listo.


## 2. Inspección de métricas — pregunta a pregunta

Antes de ejecutar el benchmark completo, es instructivo ver qué hace cada métrica
internamente. Tomamos una sola pregunta y llamamos a cada métrica de forma manual.

In [3]:
question     = "Where do the ogres from the Momotaro folktale live?"
ground_truth = "The ogres from the Momotaro folktale live in a fortified castle, known as the Ogres' Island castle."

# Preguntamos al agente
answer, context = evaluator.get_answer(question)
print(f"\nRespuesta       : {answer}")
print(f"Chunks: {context}")
print(f"Chunks recuperados: {len(context)}")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=4, column=13, offset=60>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 60, 'line': 4, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        CALL () {\n            // Vector search\n            CALL db.index.vector.queryNodes($vector_index, $top_k, $query_embedding)\n            YIELD node, score\n            \n            WITH collect({node: node, score: score}) AS nodes, max(score) AS maxScore, min(score) AS minScore\n            UNWIND nodes AS n\n            RET


Respuesta       : The ogres from the Momotaro folktale live on the Ogres' Island castle and in their village.
Chunks: ["name: the Ogres' Island castle\ndescription: a fortified residence where Momotaro fought the Ogres\ntype: castle", 'name: their village\ndescription: where Momotaro lived with his adoptive parents\ntype: village']
Chunks recuperados: 2


In [4]:
# --- context_recall ---
# Objetivo: ¿puede atribuirse cada frase del ground truth al contexto recuperado?
# Puntuación = (frases atribuidas) / (total de frases)  →  rango [0, 1]
recall_result = evaluator.evaluate_context_recall(question, ground_truth, context, verbose=True)

print("context_recall")
print(f"  puntuación : {recall_result.recall:.3f}")
print(f"  frases     : {recall_result.sentences}")
print(f"  atribuidas : {recall_result.attributions}")
print(f"  razonamiento: {recall_result.reasoning}")


── context_recall ──────────────────────────────────
  Question    : Where do the ogres from the Momotaro folktale live?
  Ground truth: The ogres from the Momotaro folktale live in a fortified castle, known as the Ogres' Island castle.
  Context chunks retrieved: 2

  [Step 1] Asking the LLM to attribute each ground-truth sentence to context...

  Ground-truth sentences (1):
    [✓ attributed] the ogres from the momotaro folktale live in a fortified castle, known as the ogres island castle.

  Reasoning : The ground truth answer is supported by the context because it mentions 'the Ogres' Island castle', which is described as a 'fortified residence where Momotaro fought the Ogres'. This directly answers the question about where the ogres live.
  Score     : 1.000
────────────────────────────────────────────────────
context_recall
  puntuación : 1.000
  frases     : ['the ogres from the momotaro folktale live in a fortified castle, known as the ogres island castle.']
  atribuidas : [1]

In [5]:
# --- faithfulness ---
# Paso 1: descomponer la respuesta en afirmaciones atómicas sin pronombres.
# Paso 2: verificar si cada afirmación puede inferirse del contexto.
# Puntuación = (afirmaciones respaldadas) / (total)  →  rango [0, 1]
faith_result = evaluator.evaluate_faithfulness(question, str(answer), context, verbose=True)

print("faithfulness")
print(f"  puntuación  : {faith_result.faithfulness:.3f}")
print(f"  afirmaciones: {faith_result.statements}")
print(f"  veredictos  : {faith_result.verdicts}")


── faithfulness ────────────────────────────────────
  Question: Where do the ogres from the Momotaro folktale live?
  Answer  : The ogres from the Momotaro folktale live on the Ogres' Island castle and in their village.

  [Step 1] Decomposing the answer into statements...
  Statements extracted (3):
    1. The ogres from the Momotaro folktale live on an island.
    2. Their home is called the Ogres' Island castle.
    3. They also reside in a village.

  [Step 2] Context detected.
           Using retrieval-grounded mode...

  Verdicts:
    [✗ unsupported] The ogres from the Momotaro folktale live on an island.
              → The statement adds unsupported new facts about their living arrangements.
    [✓ supported] Their home is called the Ogres' Island castle.
              → The context explicitly supports this statement as the description of 'the Ogres' Island castle'.
    [✓ supported] They also reside in a village.
              → The context semantically equates to this stat

In [6]:
# --- answer_correctness ---
# Se descomponen tanto la respuesta como el ground truth en afirmaciones.
# Cada afirmación se clasifica en TP / FP / FN.
# Puntuación = F1(precisión, recall) calculado a partir de los conteos TP / FP / FN.
corr_result = evaluator.evaluate_answer_correctness(question, str(answer), ground_truth, verbose=True)

print("answer_correctness")
print(f"  F1        : {corr_result.answer_correctness:.3f}")
print(f"  precisión : {corr_result.precision:.3f}")
print(f"  recall    : {corr_result.recall:.3f}")
print(f"  TP={corr_result.tp}  FP={corr_result.fp}  FN={corr_result.fn}")



── answer_correctness ──────────────────────────────
  Question    : Where do the ogres from the Momotaro folktale live?
  Answer      : The ogres from the Momotaro folktale live on the Ogres' Island castle and in their village.
  Ground truth: The ogres from the Momotaro folktale live in a fortified castle, known as the Ogres' Island castle.

  [Step 1] Decomposing the answer into statements...
  Answer statements (4):
    1. The ogres from the Momotaro folktale reside on an island.
    2. The island is called Ogres' Island.
    3. There is a castle located on Ogres' Island.
    4. In addition to living on Ogres' Island, the ogres also live in their village.

  [Step 2] Decomposing the ground truth into statements...
  Ground truth statements (2):
    1. The ogres from the Momotaro folktale reside on an island.
    2. This island is called the Ogres' Island.

  [Step 3] Classifying each statement as TP / FP / FN...
  Classifications:
    [TP] The ogres from the Momotaro folktale resi

## 3. Pipeline de benchmark completo

### Formato del dataset de benchmark

El benchmark es un CSV delimitado por punto y coma con dos columnas:
- `question` — la pregunta en lenguaje natural
- `cypher` — una consulta Cypher que se **ejecuta en tiempo de ejecución** para obtener el ground truth

Este diseño garantiza que los ground truths estén siempre sincronizados con el estado
real del grafo: no hay cadenas de texto codificadas que queden desactualizadas al cambiar los datos.

### Categorías de preguntas

| Categoría | Ejemplos | Propósito |
|---|---|---|
| **Saludos** | "Hello", "What can you do?" | Evaluar respuesta conversacional |
| **Fuera de ámbito** | "Weather?", "World Cup?" | Evaluar rechazo correcto |
| **Datos ausentes** | "Einstein's middle name?" | Evaluar honestidad del sistema |
| **Hechos simples** | "Where was Einstein born?" | Evaluar recuperación directa |
| **Consultas relacionales** | "Which institutions did Einstein work at?" | Evaluar traversal de grafo |
| **Consultas complejas** | "Where did Einstein live after Germany?" | Evaluar razonamiento multi-salto |
| **Agregaciones** | "How many Person nodes?" | Evaluar text2cypher |

El dataset inline cubre 20 preguntas; el CSV en `../data/benchmark_data.csv` cubre 38 preguntas.

> **⚠️ Requisito previo — datos del grafo**
>
> Las consultas Cypher de este benchmark usan el esquema de dominio nuevo
> (`Person`, `Location`, `ScientificTheory`, `Institution`, `Award`, `ScientificField`).
> Si el grafo todavía contiene nodos del esquema genérico antiguo (`Entity {type}`),
> las consultas devolverán resultados vacíos.
>
> Para reiniciar desde cero, ejecuta las dos celdas siguientes y luego vuelve a correr
> `01_ingestion_demo.ipynb` antes de continuar.

```python
# Vaciar la base de datos (¡irreversible!)
# neo4j.execute_query('MATCH (n) DETACH DELETE n')
# print('Base de datos vaciada — ejecuta 01_ingestion_demo.ipynb antes de continuar.')
```

In [ ]:
# Dataset de benchmark para Einstein — ground truths obtenidos dinámicamente
# vía consultas Cypher ejecutadas en tiempo de ejecución.
# Cubre todas las categorías de clase: saludos, preguntas irrelevantes,
# datos ausentes, hechos simples, consultas complejas y agregaciones.
import io

csv_content = """question;cypher
Hello;RETURN 'Hello! I am a knowledge assistant specialising in physicists and their scientific contributions. You can ask me about people like Albert Einstein, theories they developed, awards they received, institutions they worked at, and locations they lived in.' AS ground_truth
What can you do?;RETURN 'I can answer questions about physicists: theories they developed, awards they received, institutions they worked at, and locations they lived in.' AS ground_truth
What is the weather like today?;RETURN 'This question is outside my scope. I only answer questions about physicists and their scientific contributions.' AS ground_truth
Which movies did Einstein appear in?;RETURN 'This question is outside my scope. I only answer questions about physicists and their scientific contributions.' AS ground_truth
What is Einstein's middle name?;RETURN 'This information is not in the knowledge base.' AS ground_truth
Who was Einstein's wife?;RETURN 'This information is not in the knowledge base.' AS ground_truth
Where was Einstein born?;MATCH (p:Person)-[r:BORN_IN]->(l:Location) WHERE p.name CONTAINS 'Einstein' RETURN l.name + CASE WHEN r.year IS NOT NULL THEN ' (' + toString(r.year) + ')' ELSE '' END AS ground_truth ORDER BY size(l.name) DESC LIMIT 1
What is Einstein's nationality?;RETURN 'German' AS ground_truth
In what year was Einstein born?;MATCH (p:Person) WHERE p.name CONTAINS 'Einstein' OPTIONAL MATCH (p)-[r:BORN_IN]->() RETURN coalesce(toString(r.year), toString(r.year), 'This information is not in the knowledge base.') AS ground_truth LIMIT 1
What award did Einstein receive?;MATCH (p:Person)-[r:RECEIVED_AWARD]->(a:Award) WHERE p.name CONTAINS 'Einstein' RETURN a.name AS ground_truth
Why did Einstein receive the Nobel Prize?;RETURN 'Einstein received the Nobel Prize for his explanation of the photoelectric effect.' AS ground_truth
Which institutions did Einstein work at?;MATCH (p:Person)-[r:WORKED_AT]->(i:Institution) WHERE p.name CONTAINS 'Einstein' RETURN i.name + CASE WHEN r.from_year IS NOT NULL THEN ' (' + toString(r.from_year) + '-' + coalesce(toString(r.to_year), '?') + ')' ELSE '' END AS ground_truth ORDER BY r.from_year
What theories did Einstein develop?;MATCH (p:Person)-[r:DEVELOPED]->(t:ScientificTheory) WHERE p.name CONTAINS 'Einstein' RETURN t.name + CASE WHEN r.year IS NOT NULL THEN ' (' + toString(r.year) + ')' ELSE '' END AS ground_truth ORDER BY r.year
In what year was the theory of general relativity published?;RETURN '1915' AS ground_truth
What scientific fields did Einstein contribute to?;MATCH (p:Person)-[:CONTRIBUTED_TO]->(f:ScientificField) WHERE p.name CONTAINS 'Einstein' RETURN f.name AS ground_truth ORDER BY f.name
Where did Einstein live after leaving Germany?;MATCH (p:Person)-[r:LIVED_IN]->(l:Location) WHERE p.name CONTAINS 'Einstein' RETURN l.name + CASE WHEN r.from_year IS NOT NULL THEN ' (from ' + toString(r.from_year) + ')' ELSE '' END AS ground_truth ORDER BY r.from_year
Which Person nodes are connected to Princeton?;MATCH (p:Person)-[:LIVED_IN|WORKED_AT]->(n) WHERE toLower(n.name) CONTAINS 'princeton' RETURN DISTINCT p.name AS ground_truth
How many Person nodes are in the graph?;MATCH (p:Person) RETURN toString(count(p)) + ' person(s)' AS ground_truth
How many Institution nodes are in the graph?;MATCH (i:Institution) RETURN toString(count(i)) + ' institution(s)' AS ground_truth
How many typed relationships exist in the graph?;MATCH ()-[r]->() WHERE type(r) IN ['BORN_IN','WORKED_AT','LIVED_IN','DEVELOPED','PUBLISHED','RECEIVED_AWARD','CONTRIBUTED_TO'] RETURN toString(count(r)) + ' relationship(s)' AS ground_truth
"""

dataset = pd.read_csv(io.StringIO(csv_content), delimiter=";")
print(f"Dataset de benchmark: {len(dataset)} preguntas")
dataset[['question']]

### 3.1 `run_benchmark()` — ejecutar ground truths Cypher y llamar al agente

Para cada fila:
1. Ejecuta la consulta Cypher → string con el ground truth
2. Llama a `rag.answer(question)` → respuesta del agente + contextos recuperados
3. Registra la latencia en segundos

In [ ]:
results_df = evaluator.run_benchmark(dataset, verbose=True)

print(f"\nBenchmark completo — {len(results_df)} filas")
results_df[['question', 'ground_truth', 'answer', 'latency']]

### 3.2 `evaluate_results()` — puntuar cada fila con las tres métricas RAGAS

Este paso es el más intensivo en LLM: cada fila hace ~5 llamadas al LLM
(context_recall × 1, faithfulness × 2, answer_correctness × 3).
Esperar unos 10–30 segundos por fila dependiendo del modelo.

In [ ]:
scored_df = evaluator.evaluate_results(results_df, verbose=True)

print(f"\nPuntuación completada.")
scored_df[['question', 'context_recall', 'faithfulness', 'answer_correctness', 'latency']]

### 3.3 `print_summary()` — resultados agregados

El resumen reproduce la tabla de resumen de benchmark que hemos comentado en clase.
Un sistema bien ajustado debería superar 0.7 en las tres métricas.

In [ ]:
evaluator.print_summary(scored_df)

## 4. Análisis de resultados

### 4.1 Distribución por métrica

In [ ]:
import matplotlib.pyplot as plt

metrics = ['context_recall', 'faithfulness', 'answer_correctness']
colors  = ['steelblue', 'seagreen', 'darkorange']

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle('Distribución de métricas RAGAS — benchmark Einstein', fontsize=13)

for ax, metric, color in zip(axes, metrics, colors):
    ax.hist(scored_df[metric].dropna(), bins=5, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(scored_df[metric].mean(), color='black', linestyle='--', linewidth=1.2,
               label=f'media={scored_df[metric].mean():.2f}')
    ax.set_title(metric)
    ax.set_xlabel('Puntuación')
    ax.set_xlim(0, 1)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Distribución de latencia
fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(range(len(scored_df)), scored_df['latency'], color='slategray')
ax.set_yticks(range(len(scored_df)))
ax.set_yticklabels(
    [q[:40] + '...' if len(q) > 40 else q for q in scored_df['question']],
    fontsize=8
)
ax.set_xlabel('Latencia (s)')
ax.set_title('Latencia de respuesta por pregunta')
plt.tight_layout()
plt.show()

### 4.2 Mejor y peor caso por answer_correctness

Inspeccionar los extremos ayuda a diagnosticar *por qué* falla el sistema:
- `context_recall` bajo → el recuperador no encontró los chunks relevantes → ajustar `chunk_size` o `top_k`
- `faithfulness` bajo → el LLM alucinó → mejorar el prompt de respuesta o añadir instrucciones de anclaje
- `answer_correctness` bajo con `faithfulness` alto → el contexto recuperado era incompleto

In [ ]:
def show_case(label, row):
    print(f"{'='*70}")
    print(label)
    print(f"{'='*70}")
    print(f"Question        : {row['question']}")
    print(f"Ground truth    : {str(row['ground_truth'])[:200]}")
    print(f"Answer          : {str(row['answer'])[:200]}")
    print(f"answer_correctness : {row['answer_correctness']:.3f}")
    print(f"context_recall     : {row['context_recall']:.3f}")
    print(f"faithfulness       : {row['faithfulness']:.3f}")
    print()

best_idx  = scored_df['answer_correctness'].idxmax()
worst_idx = scored_df['answer_correctness'].idxmin()

show_case("MEJOR CASO",  scored_df.loc[best_idx])
show_case("PEOR CASO", scored_df.loc[worst_idx])

### 4.3 Correlación entre métricas

¿Son las tres métricas independientes o tienden a moverse juntas?
Una correlación fuerte entre `context_recall` y `answer_correctness` confirmaría
que la calidad de la recuperación es el principal cuello de botella.

In [ ]:
corr = scored_df[metrics].corr()
print("Correlación de Pearson entre métricas:")
print(corr.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].scatter(scored_df['context_recall'], scored_df['answer_correctness'],
                color='steelblue', edgecolors='white', s=80, alpha=0.85)
axes[0].set_xlabel('context_recall')
axes[0].set_ylabel('answer_correctness')
axes[0].set_title('Recall de contexto vs Corrección de respuesta')
axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1)

axes[1].scatter(scored_df['faithfulness'], scored_df['answer_correctness'],
                color='seagreen', edgecolors='white', s=80, alpha=0.85)
axes[1].set_xlabel('faithfulness')
axes[1].set_ylabel('answer_correctness')
axes[1].set_title('Fidelidad vs Corrección de respuesta')
axes[1].set_xlim(0, 1); axes[1].set_ylim(0, 1)

plt.suptitle('Correlaciones entre métricas', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Uso de un fichero CSV de benchmark

Para una evaluación real se mantiene un CSV versionado junto al código.
El formato es idéntico — `load_dataset()` simplemente lo lee desde disco.
Las consultas Cypher se adaptan automáticamente a los datos actuales del grafo.

In [ ]:
# Cargar el CSV de benchmark del proyecto (preguntas genéricas sobre la estructura del grafo)
generic_dataset = evaluator.load_dataset('../data/benchmark_data.csv')
print(f"Benchmark CSV: {len(generic_dataset)} preguntas")
generic_dataset[['question']]

In [ ]:
# Ejecutar el pipeline completo sobre el benchmark CSV
# (descomenta si quieres correrlo — tarda varios minutos)

# generic_results = evaluator.run_benchmark(generic_dataset)
# generic_scored  = evaluator.evaluate_results(generic_results)
# evaluator.print_summary(generic_scored)


## 6. Guardar resultados

In [ ]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = f"../data/evaluation_results_{timestamp}.csv"
scored_df.to_csv(out_path, index=False)
print(f"Resultados guardados en {out_path}")


In [ ]:
neo4j.close()
